# Netflix Content Strategy: A Deep Dive EDA

### Dataset Description
The dataset contains a list of all movies and TV shows available on Netflix as of 2021, including details such as:
- **Show ID**: Unique identifier for every movie/TV show.
- **Type**: Identifier - A Movie or TV Show.
- **Title**: Title of the movie/TV show.
- **Director**: Director of the movie.
- **Cast**: Actors involved in the movie/show.
- **Country**: Country where the movie/show was produced.
- **Date Added**: Date it was added on Netflix.
- **Release Year**: Actual Release year of the movie/show.
- **Rating**: TV Rating of the movie/show.
- **Duration**: Total Duration - in minutes or number of seasons.
- **Listed In**: Genre.
- **Description**: The summary description.

### Objective
This analysis aims to uncover Netflix's content strategy by examining:
1. **Data Cleaning**: Handling missing values and formatting dates.
2. **Univariate Analysis**: Understanding the distribution of content types and ratings.
3. **Temporal Trends**: Tracking content growth over the years.
4. **Geographical Distribution**: Identifying key production hubs.
5. **Genre Insights**: Deep dive into content categories and viewer preferences.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_style("white")
netflix_red = "#E50914"
dark_grey = "#221F1F"

# Load dataset
df = pd.read_csv('netflix_titles.csv')
print("Libraries imported and dataset loaded successfully.")

In [ ]:
print("--- Dataset Info ---")
df.info()
print("\n--- First 5 Rows ---")
display(df.head())

## 2.1 Data Quality Audit
Before cleaning, we need to understand the extent of missing data and potential redundancies.

In [ ]:
print("--- Missing Values Summary ---")
missing_data = df.isnull().sum().sort_values(ascending=False)
percent_missing = (df.isnull().sum()/df.isnull().count()*100).sort_values(ascending=False)
missing_df = pd.concat([missing_data, percent_missing], axis=1, keys=['Total', 'Percent'])
display(missing_df)

print("\n--- Duplicate Rows ---")
print(f"Number of duplicate entries: {df.duplicated().sum()}")

In [ ]:
print("--- Statistical Profiling (Categorical) ---")
display(df.describe(include='object').T)

print("\n--- Statistical Profiling (Numeric) ---")
display(df.describe().T)

In [ ]:
print("--- Unique Values & Cardinality ---")
for col in df.columns:
    print(f"{col:20} : {df[col].nunique()} unique values")

In [ ]:
# Handling missing values
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Unknown')
df['country'] = df['country'].fillna(df['country'].mode()[0])
df['rating'] = df['rating'].fillna(df['rating'].mode()[0])

# Dropping rows with missing date_added or duration
df.dropna(subset=['date_added', 'duration'], inplace=True)

# Convert date_added to datetime objects
df['date_added'] = pd.to_datetime(df['date_added'].str.strip())
print("Data cleaning complete: Missing values handled and dates converted.")

In [ ]:
# Feature Extraction
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month
print("Year and Month extracted from date_added.")

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(x='type', data=df, palette=[netflix_red, dark_grey])
plt.title('Content Type Distribution: Movies vs TV Shows', fontsize=16, fontweight='bold', color=dark_grey)
plt.xlabel('Type', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.show()

### Analysis of the Content Mix Ratio
**Technical Detail**: This visualization uses a simple count plot to compare the volume of Movies vs. TV Shows in the dataset.

**Business Intuition**: Netflix's catalog is significantly dominated by Movies. This suggests that while TV shows are vital for long-term engagement (binge-watching), movies remain a critical component for attracting a broad audience with shorter time commitments. For a recommendation engine, knowing the user's preference for 'one-off' vs 'serial' content is a primary feature for high-accuracy suggestions.

In [ ]:
plt.figure(figsize=(12, 6))
top_10_countries = df['country'].value_counts()[:10]
sns.barplot(x=top_10_countries.values, y=top_10_countries.index, palette='Reds_r')
plt.title('Top 10 Content Producing Countries', fontsize=16, fontweight='bold', color=dark_grey)
plt.xlabel('Number of Titles', fontsize=12)
plt.show()

### Geographical Dominance in the Streaming Library
**Technical Detail**: We analyzed the `country` field to identify the top 10 contributors to the Netflix library.

**Business Intuition**: The United States is the clear leader, followed by India. This highlights Netflix's dual focus: maintaining its home-market dominance while aggressively catering to the massive Indian film industry. From a content acquisition perspective, this data suggests that localized content from these top regions is a high-yield investment, as they have established production pipelines that scale well on global platforms.

In [ ]:
plt.figure(figsize=(12, 6))
content_by_year = df.groupby(['year_added', 'type']).size().reset_index(name='count')
sns.lineplot(data=content_by_year, x='year_added', y='count', hue='type', palette=[netflix_red, dark_grey], linewidth=2.5)
plt.title('Content Addition Over Time', fontsize=16, fontweight='bold', color=dark_grey)
plt.xlabel('Year Added', fontsize=12)
plt.ylabel('Number of Titles', fontsize=12)
plt.show()

### Netflix’s Exponential Growth and the Digital Shift
**Technical Detail**: A temporal line plot tracking content additions annually, segmented by content type.

**Business Intuition**: There is a massive inflection point around 2015-2016, where content addition accelerated exponentially. This corresponds with Netflix's aggressive transition into producing 'Netflix Originals.' The slight decline in recent years might indicate a shift in strategy from 'quantity' to 'quality,' or perhaps the impact of global production halts. This trend is vital for capacity planning in content delivery networks (CDNs).

In [ ]:
plt.figure(figsize=(12, 6))
order = df['rating'].value_counts().index
sns.countplot(x='rating', data=df, order=order, palette='Reds_r')
plt.title('Distribution of Content Ratings', fontsize=16, fontweight='bold', color=dark_grey)
plt.xticks(rotation=45)
plt.show()

### Target Audience Demographics
**Technical Detail**: This bar chart shows the distribution of content maturity ratings.

**Business Intuition**: The dominance of `TV-MA` and `TV-14` indicates that Netflix’s primary audience is mature adults and teenagers. This rating profile is critical for maintaining 'Brand Safety' and ensuring that recommendation algorithms respect age-appropriateness, which is a major legal and ethical requirement in the streaming industry.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(18, 6))

# Movie Durations
movie_durations = df[df['type'] == 'Movie']['duration'].str.replace(' min', '').astype(int)
sns.histplot(movie_durations, bins=30, kde=True, color=netflix_red, ax=ax[0])
ax[0].set_title('Movie Duration Distribution', fontsize=14, fontweight='bold')
ax[0].set_xlabel('Minutes')

# TV Show Seasons
tv_seasons = df[df['type'] == 'TV Show']['duration'].str.replace(' Season', '').str.replace('s', '').astype(int)
sns.countplot(x=tv_seasons, color=dark_grey, ax=ax[1])
ax[1].set_title('TV Show Season Distribution', fontsize=14, fontweight='bold')
ax[1].set_xlabel('Number of Seasons')

plt.tight_layout()
plt.show()

### Viewer Preference: Long-form vs. Short-form Content
**Technical Detail**: Combined histogram (for movies) and count plot (for TV show seasons).

**Business Intuition**: Most movies are around 90-100 minutes, which is the sweet spot for viewer attention span. Interestingly, the majority of TV shows have only 1 season. This suggests a 'fail-fast' content strategy where Netflix tests many new concepts and only invests in subsequent seasons for high-performing titles. This is a classic example of data-driven production decisions.

In [ ]:
from collections import Counter

# Parsing comma-separated genres
genres = df['listed_in'].str.split(', ')
genre_counts = Counter([genre for sublist in genres for genre in sublist])
top_genres = pd.DataFrame(genre_counts.most_common(10), columns=['Genre', 'Count'])

plt.figure(figsize=(12, 6))
sns.barplot(x='Count', y='Genre', data=top_genres, palette='Reds_r')
plt.title('Top 10 Genres on Netflix', fontsize=16, fontweight='bold', color=dark_grey)
plt.show()

### Genre Popularity and Niche Content Trends
**Technical Detail**: We 'exploded' the listed_in column to count individual genre occurrences accurately.

**Business Intuition**: 'International Movies', 'Dramas', and 'Comedies' are the leading genres. The high volume of 'International' titles confirms Netflix's global expansion strategy. For a Senior Engineer, this genre data is the foundation for 'Content-Based Filtering' in the recommendation engine, allowing the system to map users to their preferred storytelling styles.

# Overall Strategic Insights

### Key Findings
1. **Catalog Composition**: Netflix maintains a massive volume of movies, but the growth in original TV shows is the primary driver for subscriber retention.
2. **Global Strategy**: The dominance of the US and India indicates a highly targeted regional strategy, leveraging established cinematic hubs.
3. **Experimental Production**: The high frequency of single-season shows highlights a strategy of rapid experimentation and data-driven renewals.
4. **Maturity Alignment**: The library is clearly skewed toward adult and teenage demographics, distinguishing Netflix from more family-oriented competitors.

### Senior Engineer Perspective
From a Machine Learning perspective, this dataset highlights the importance of metadata-driven recommendation systems. Features like `country`, `rating`, and `listed_in` are high-signal predictors of user interest. Furthermore, the temporal trends suggest that 'Freshness' (content recently added) should be heavily weighted in the 'Recency' component of a ranking algorithm. To optimize content acquisition, Netflix should continue using these distribution patterns to identify 'content gaps'—genres or regions that are underrepresented but have high engagement potential based on user behavior data.